# Лаба 3: партиционирование и эволюция схемыВ этом ноутбуке мы посмотрим, как Iceberg работает с партиционированными таблицами и эволюцией схемы поверх общего каталога `lakehouse`.Перед началом убедись, что стенд запущен (`docker compose up -d`) и Spark-кластер доступен.

In [ ]:
from pyspark.sql import SparkSessionspark = (    SparkSession.builder    .appName("lab3-partitioning-schema-evolution")    .master("spark://spark-master:7077")    .getOrCreate())

## Часть 1. Партиционированная таблицаДля начала создадим партиционированную Iceberg-таблицу `lakehouse.default.partition_demo` с помощью готового SQL-скрипта из `src/spark/partitioned_table_demo.sql`.

In [ ]:
from pathlib import Pathsql_path = Path("/opt/src/spark/partitioned_table_demo.sql")sql_text = sql_path.read_text(encoding="utf-8")for statement in sql_text.split(";"):    stmt = statement.strip()    if stmt:        spark.sql(stmt)

Посмотрим, какие данные записаны по датам, и обсудим партиционирование по `event_date`.

In [ ]:
spark.sql("""SELECT  event_date,  COUNT(*) AS cnt,  SUM(amount) AS total_amountFROM lakehouse.default.partition_demoGROUP BY event_dateORDER BY event_date""").show()

In [ ]:
spark.sql("""SELECT *FROM lakehouse.default.partition_demoWHERE event_date = DATE '2024-01-01'ORDER BY user_id""").show()

## Часть 2. Эволюция схемыТеперь посмотрим на эволюцию схемы: создадим таблицу, добавим колонку и вставим новые строки с дополнительными данными.Для подготовки таблицы используем скрипт `src/spark/schema_evolution_demo.sql`.

In [ ]:
sql_path = Path("/opt/src/spark/schema_evolution_demo.sql")sql_text = sql_path.read_text(encoding="utf-8")for statement in sql_text.split(";"):    stmt = statement.strip()    if stmt:        spark.sql(stmt)

In [ ]:
spark.sql("DESCRIBE TABLE lakehouse.default.schema_evolution_demo").show(truncate=False)

In [ ]:
spark.sql("""SELECT *FROM lakehouse.default.schema_evolution_demoORDER BY id""").show(truncate=False)

Обрати внимание, что старые строки имеют `NULL` в колонке `metadata`, а новые — заполненное значение.Iceberg хранит историю снапшотов и позволяет эволюцию схемы без сложных миграций.